In [1]:
# ── CELL 1 : Install Unsloth and dependencies ─────────────────────────────

%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

print('✅  All packages installed successfully.')


In [2]:
# ── CELL 2 : Config & Imports ─────────────────────────────────────────────

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import login
import torch

# ── EDIT THESE ───────────────────────────────────────────────
HF_DATASET_ID = "YOUR_HF_REPO_NAME"  # Your dataset from Stage 1
HF_TOKEN      = "YOUR_HF_TOKEN"        # https://huggingface.co/settings/tokens (Ensure WRITE access)
NEW_MODEL_ID  = "YOUR_DESTINATION_MODEL_FOLDER" # Where to save your fine-tuned adapters

max_seq_length = 1024 # Perfect context window for retrieving long legal citations
dtype = None # None auto-detects (usually float16 for T4)
load_in_4bit = True # Essential to fit Mistral-7b on a Free 16GB GPU

print('✅  Config loaded.')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅  Config loaded.


In [3]:
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1' # Forces Unsloth to skip HuggingFace entirely
os.environ["UNSLOTH_RETURN_STATISTICS"] = "0" # Kills the Unsloth analytics ping that caused the timeout

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print("📥 Downloading Mistral-7B-Instruct from ModelScope...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    token = "YOUR_HF_REPO_NAME", # Make sure to quickly paste your token back in
)

# Essential: Setup the ChatML template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

print('\n✅  Base Model and Tokenizer loaded successfully!')


📥 Downloading Mistral-7B-Instruct from ModelScope...
==((====))==  Unsloth 2026.3.18: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will map <|im_end|> to EOS = </s>.



✅  Base Model and Tokenizer loaded successfully!


In [4]:
# ── CELL 4 : Add LoRA Adapters ────────────────────────────────────────────

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: 16 is a solid balance between learning capacity and VRAM usage.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # Targeting all layers ensures deeper learning
    lora_alpha = 16,
    lora_dropout = 0, # Dropout = 0 is optimized for Unsloth specifically
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print('✅  LoRA adapters injected! Ready for training.')


Unsloth 2026.3.18 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅  LoRA adapters injected! Ready for training.


In [5]:
# ── CELL 5 : Load Dataset ─────────────────────────────────────────────────

# Autenticate first (in case repo_lawgpt is private)
login(token=HF_TOKEN)

print(f"📥 Loading dataset: {HF_DATASET_ID}...")
dataset = load_dataset(HF_DATASET_ID, split="train")

print(f"\n📊 Loaded {len(dataset)} training rows.")
print("\nPreview of the first row:")
print("="*60)
print(dataset[0]["text"][:600] + "...\n")
print("="*60)


📥 Loading dataset: SCARA02/repo_lawgpt...


README.md:   0%|          | 0.00/493 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/881k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/113k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1231 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/137 [00:00<?, ? examples/s]


📊 Loaded 1231 training rows.

Preview of the first row:
<|im_start|>system
You are LawGPT-IN, an expert AI assistant specialising in Indian law. You have deep knowledge of the Indian Penal Code, CrPC, Constitution of India, and landmark Supreme Court and High Court judgements. Always cite the relevant acts and sections in your answers. Respond clearly and in plain language that non-lawyers can understand.<|im_end|>
<|im_start|>user
Based on the following facts, what was the court's verdict?

**Facts:**
, the

rival submissions made across the Bar, and the materials

placed on record, particularly on considering the fact that

the petitioners have b...



In [7]:
# ── CELL 6 : Setup Trainer & Start Fine-tuning ────────────────────────────

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set to False since standard Q&A ChatML pairs vary wildly in length
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 5, # ⚠️ FOR DEMO/TESTING: Set to 60 steps (~15 mins) to verify everything works.
        # For a full production run: comment out `max_steps` above, and uncomment `num_train_epochs = 1` below:
        # num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)


print('🚀  Starting training...\n')
trainer_stats = trainer.train()


🚀  Starting training...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,231 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss
1,1.846155
2,1.720174
3,1.781054
4,1.363896
5,1.401161


In [8]:
# ── CELL 7 : Test the Model ────────────────────────────────────────────────

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Try asking an Indian-specific legal question:
messages = [
    {"role": "system", "content": "You are LawGPT-IN, an expert AI assistant specialising in Indian law. Always cite the relevant acts."},
    {"role": "user", "content": "Can you explain the difference between Anticipatory Bail and Regular Bail under the CrPC?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,  # Adds <|im_start|>assistant\n
    return_tensors = "pt",
).to("cuda")

print("🤖 Generating response...\n")
outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
response = tokenizer.batch_decode(outputs)

print("\n" + "="*60 + "\nRESULTS:\n")

# Clean the output string to easily isolate the assistant's reply
raw_str = response[0]
if "<|im_start|>assistant\n" in raw_str:
    final_reply = raw_str.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
else:
    final_reply = raw_str
print(final_reply)
print("\n" + "="*60)


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 Generating response...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


RESULTS:

Absolutely, I'd be happy to help explain the difference between Anticipatory Bail and Regular Bail under the CrPC (Code of Criminal Procedure).

Regular Bail: This is a bail granted to a person who has already been arrested and is in judicial custody. It is granted to ensure that the accused is not unnecessarily detained in custody and is released on bail until the trial is completed.

Anticipatory Bail: This is a bail granted to a person who has not been arrested yet but fears arrest in a non-bailable offence. It is granted to prevent the person from being arrested and detained in custody for a prolonged period while the investigation is ongoing. The person seeking anticipatory bail must show that there is a prima facie case against them and that they are likely to be falsely implicated.

In summary, Regular Bail is for those who have already been arrested, and Anticipatory Bail is for those who have not been arrested but fear arrest.



In [9]:
# ── CELL 8 : Save & Push Models to HuggingFace Hub ────────────────────────

print(f"⬆️  Pushing LoRA adapters to {NEW_MODEL_ID}...")

# This pushes just the ~100MB adapter layers, NOT the entire 7GB model.
model.push_to_hub_merged(NEW_MODEL_ID, tokenizer, save_method = "lora", token = HF_TOKEN)

print(f"\n✅  SUCCESS! Your fine-tuned model adapters are securely uploaded to:")
print(f"   https://huggingface.co/{NEW_MODEL_ID}")
print("\n🎯  Next step: Stage 3 — Hybrid RAG + UI Deployment")


⬆️  Pushing LoRA adapters to SCARA02/lawgpt-mistral-7b-v1...


config.json:   0%|          | 0.00/722 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0003-of-00003.safetensors:   3%|3         |  152MB / 4.54GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [10:58<00:00, 219.61s/it]


Unsloth: Merge process complete. Saved to `/content/SCARA02/lawgpt-mistral-7b-v1`

✅  SUCCESS! Your fine-tuned model adapters are securely uploaded to:
   https://huggingface.co/SCARA02/lawgpt-mistral-7b-v1

🎯  Next step: Stage 3 — Hybrid RAG + UI Deployment


In [10]:
# ── CELL 9 : Load test datasets ───────────────────────────────
# Source 1: Your own test split (same format as training)
# Source 2: law-ai/InLegalNLP (out-of-distribution legal QA)

import re
from datasets import load_dataset

# ── Source 1: Your own test split ─────────────────────────────
print('📥 Loading your test split...')
own_test = load_dataset(HF_DATASET_ID, split="test", token=HF_TOKEN)
print(f'   Own test split: {len(own_test)} rows')

# Parse ChatML text back into question/answer pairs for evaluation
def parse_chatml(text):
    """
    Extract user question and assistant reference answer
    from a ChatML-formatted training row.
    """
    user_match = re.search(
        r'<\|im_start\|>user\n(.*?)<\|im_end\|>', text, re.DOTALL
    )
    asst_match = re.search(
        r'<\|im_start\|>assistant\n(.*?)<\|im_end\|>', text, re.DOTALL
    )
    return {
        "question":  user_match.group(1).strip() if user_match else "",
        "reference": asst_match.group(1).strip() if asst_match else "",
    }

own_pairs = [parse_chatml(row["text"]) for row in own_test]
own_pairs = [p for p in own_pairs if p["question"] and p["reference"]]
print(f'   Parsed {len(own_pairs)} valid Q&A pairs from test split')

# ── Source 2: InLegalNLP (out-of-distribution) ────────────────
print('\n📥 Loading law-ai/InLegalNLP...')
try:
    inlegal = load_dataset("law-ai/InLegalNLP", split="test")
    print(f'   InLegalNLP columns: {inlegal.column_names}')
    print(f'   InLegalNLP rows   : {len(inlegal)}')
    HAS_INLEGAL = True
except Exception as e:
    print(f'   ⚠️  Could not load InLegalNLP: {e}')
    print('   Will use ILDC as fallback.')
    HAS_INLEGAL = False

# ── Fallback: ILDC summarisation dataset ──────────────────────
if not HAS_INLEGAL:
    print('\n📥 Loading ILDC (fallback)...')
    try:
        ildc = load_dataset("viber1/ildc", "ildc_single", split="test")
        print(f'   ILDC columns: {ildc.column_names}')
        print(f'   ILDC rows   : {len(ildc)}')
    except Exception as e:
        print(f'   ⚠️  ILDC also failed: {e}')
        print('   Will evaluate on own test split only.')

print('\n✅  Datasets ready for evaluation.')

📥 Loading your test split...
   Own test split: 137 rows
   Parsed 137 valid Q&A pairs from test split

📥 Loading law-ai/InLegalNLP...
   ⚠️  Could not load InLegalNLP: Dataset 'law-ai/InLegalNLP' doesn't exist on the Hub or cannot be accessed.
   Will use ILDC as fallback.

📥 Loading ILDC (fallback)...
   ⚠️  ILDC also failed: Dataset 'viber1/ildc' doesn't exist on the Hub or cannot be accessed.
   Will evaluate on own test split only.

✅  Datasets ready for evaluation.


In [11]:
# ── CELL 10 : Curated Indian legal QA test suite ──────────────
# 20 questions covering all 4 pair types from your training data.
# Reference answers are grounded in actual Indian law.

CURATED_TEST = [
    # ── Bail ──────────────────────────────────────────────────
    {
        "id": "bail_01",
        "type": "legal_concept",
        "question": "What is the difference between Anticipatory Bail and Regular Bail under CrPC?",
        "reference": (
            "Anticipatory Bail is granted under Section 438 CrPC before arrest, "
            "when a person apprehends arrest for a non-bailable offence. "
            "Regular Bail is granted under Section 437 or 439 CrPC after arrest "
            "and during the pendency of trial. Key difference: anticipatory bail "
            "is pre-arrest protection; regular bail is post-arrest release."
        ),
        "key_acts": ["Section 438 CrPC", "Section 437 CrPC", "Section 439 CrPC"],
    },
    {
        "id": "bail_02",
        "type": "legal_concept",
        "question": "On what grounds can a bail application be rejected by a Sessions Court?",
        "reference": (
            "Under Section 437 CrPC, bail may be rejected if the offence is punishable "
            "by death or life imprisonment, if the accused is a repeat offender, "
            "if there is reasonable ground to believe guilt, or if bail would obstruct justice. "
            "Courts also consider flight risk and tampering with evidence."
        ),
        "key_acts": ["Section 437 CrPC"],
    },

    # ── Property ──────────────────────────────────────────────
    {
        "id": "prop_01",
        "type": "legal_concept",
        "question": "What are the legal remedies available for a property encroachment dispute in India?",
        "reference": (
            "Remedies include: filing a civil suit for injunction and declaration of title "
            "under the Specific Relief Act 1963; filing a complaint under Section 441 IPC "
            "for criminal trespass; approaching the Revenue Court for mutation disputes; "
            "or filing a writ petition if a government body is involved."
        ),
        "key_acts": ["Specific Relief Act 1963", "Section 441 IPC"],
    },
    {
        "id": "prop_02",
        "type": "legal_concept",
        "question": "What is adverse possession and how many years does it take to claim property through it in India?",
        "reference": (
            "Adverse possession allows a person who has been in continuous, open, "
            "hostile, and exclusive possession of land for a statutory period to claim title. "
            "Under the Limitation Act 1963, the period is 12 years for private land "
            "and 30 years for government land. The possession must be to the knowledge of the "
            "true owner and without the owner's permission."
        ),
        "key_acts": ["Limitation Act 1963"],
    },

    # ── Family / Divorce ──────────────────────────────────────
    {
        "id": "fam_01",
        "type": "legal_concept",
        "question": "What is Section 498A IPC and what are its key elements?",
        "reference": (
            "Section 498A IPC deals with cruelty by husband or relatives of husband. "
            "Key elements: (1) the woman must be married; (2) she must be subjected to "
            "cruelty or harassment; (3) the cruelty must be by the husband or his relatives. "
            "'Cruelty' includes both physical and mental harm, and demands for dowry. "
            "It is a cognizable, non-bailable, and non-compoundable offence."
        ),
        "key_acts": ["Section 498A IPC"],
    },
    {
        "id": "fam_02",
        "type": "legal_concept",
        "question": "Under the Hindu Marriage Act, what are the grounds for divorce?",
        "reference": (
            "Under Section 13 of the Hindu Marriage Act 1955, grounds for divorce include: "
            "adultery, cruelty, desertion for 2+ years, conversion to another religion, "
            "unsoundness of mind, leprosy, venereal disease, renunciation of the world, "
            "and presumption of death. Mutual consent divorce is available under Section 13B "
            "after 1 year of separation."
        ),
        "key_acts": ["Section 13 Hindu Marriage Act 1955", "Section 13B Hindu Marriage Act"],
    },

    # ── Consumer ──────────────────────────────────────────────
    {
        "id": "cons_01",
        "type": "legal_concept",
        "question": "What is the pecuniary jurisdiction of District, State, and National Consumer Forums under the Consumer Protection Act 2019?",
        "reference": (
            "Under the Consumer Protection Act 2019: "
            "District Commission: complaints up to Rs. 1 crore; "
            "State Commission: complaints from Rs. 1 crore to Rs. 10 crore; "
            "National Commission: complaints above Rs. 10 crore. "
            "These limits were revised from the 1986 Act values."
        ),
        "key_acts": ["Consumer Protection Act 2019"],
    },

    # ── Cheque Bounce ─────────────────────────────────────────
    {
        "id": "nigo_01",
        "type": "legal_concept",
        "question": "What is the procedure for filing a cheque bounce case under Section 138 of the Negotiable Instruments Act?",
        "reference": (
            "Procedure: (1) Cheque is dishonoured; (2) Within 30 days of receiving "
            "bank memo, send a legal demand notice to the drawer; (3) Drawer must "
            "repay within 15 days of receiving notice; (4) If no payment, file complaint "
            "in Magistrate Court within 30 days of notice period expiry. "
            "Punishment: imprisonment up to 2 years or fine up to twice the cheque amount."
        ),
        "key_acts": ["Section 138 Negotiable Instruments Act"],
    },

    # ── Constitutional / Writ ─────────────────────────────────
    {
        "id": "const_01",
        "type": "legal_concept",
        "question": "What are the five types of writs under Article 32 of the Indian Constitution?",
        "reference": (
            "The five writs under Article 32 (Supreme Court) and Article 226 (High Courts) are: "
            "1. Habeas Corpus — produce the body, used against illegal detention; "
            "2. Mandamus — command a public authority to perform its duty; "
            "3. Prohibition — prevent lower court from exceeding jurisdiction; "
            "4. Certiorari — quash a decision of lower court/authority; "
            "5. Quo Warranto — challenge a person's right to hold public office."
        ),
        "key_acts": ["Article 32 Constitution of India", "Article 226 Constitution of India"],
    },
    {
        "id": "const_02",
        "type": "legal_concept",
        "question": "What fundamental rights are guaranteed under Article 21 of the Indian Constitution?",
        "reference": (
            "Article 21 guarantees the right to life and personal liberty. "
            "The Supreme Court has expanded this to include: right to live with dignity, "
            "right to livelihood, right to health, right to education (now Article 21A), "
            "right to speedy trial, right to a clean environment, and right to privacy "
            "(held in K.S. Puttaswamy v. Union of India, 2017)."
        ),
        "key_acts": ["Article 21 Constitution of India"],
    },

    # ── Labour ────────────────────────────────────────────────
    {
        "id": "lab_01",
        "type": "legal_concept",
        "question": "What are the rights of an employee against wrongful termination under Indian labour law?",
        "reference": (
            "Under the Industrial Disputes Act 1947, a workman (employed for 1+ year "
            "in establishment with 100+ workers) cannot be retrenched without: "
            "(1) 3 months notice or pay in lieu; (2) Government permission; "
            "(3) retrenchment compensation at 15 days wages per year of service. "
            "Wrongful termination can be challenged via Labour Court under Section 2A "
            "of the Act. Service rules and standing orders also apply."
        ),
        "key_acts": ["Industrial Disputes Act 1947", "Section 2A"],
    },

    # ── Motor Accident ────────────────────────────────────────
    {
        "id": "mact_01",
        "type": "legal_concept",
        "question": "How is compensation calculated in a Motor Accident Claim Tribunal (MACT) case in India?",
        "reference": (
            "Under the Motor Vehicles Act 1988, compensation is calculated using the "
            "Multiplier Method (for death/permanent disability) as per the Supreme Court's "
            "formula in Sarla Verma v. DTC: "
            "Compensation = Annual Income x Multiplier (based on age). "
            "Components include: loss of income, medical expenses, pain and suffering, "
            "loss of consortium, and funeral expenses. No-fault liability applies "
            "under Section 163A for structured compensation."
        ),
        "key_acts": ["Motor Vehicles Act 1988", "Section 163A"],
    },

    # ── Income Tax ────────────────────────────────────────────
    {
        "id": "tax_01",
        "type": "legal_concept",
        "question": "What is the time limit for filing an appeal before the Income Tax Appellate Tribunal (ITAT)?",
        "reference": (
            "Under Section 253 of the Income Tax Act 1961, an appeal to ITAT must be "
            "filed within 60 days of the order of the Commissioner of Income Tax (Appeals). "
            "The Tribunal has powers to condone delay under Section 253(5) if sufficient "
            "cause is shown. The appeal form is Form 36 and must be accompanied by the "
            "prescribed fee."
        ),
        "key_acts": ["Section 253 Income Tax Act 1961"],
    },

    # ── Criminal ──────────────────────────────────────────────
    {
        "id": "crim_01",
        "type": "legal_concept",
        "question": "What is the difference between cognizable and non-cognizable offences under CrPC?",
        "reference": (
            "A cognizable offence (Schedule I CrPC) is one where police can arrest "
            "without a warrant, investigate without Magistrate's permission, e.g., murder, rape. "
            "A non-cognizable offence requires a warrant for arrest and "
            "Magistrate's permission to investigate, e.g., assault, cheating under certain thresholds. "
            "FIR is registered for cognizable offences; a complaint is filed for non-cognizable ones."
        ),
        "key_acts": ["CrPC Schedule I", "Section 2(c) CrPC"],
    },
    {
        "id": "crim_02",
        "type": "legal_concept",
        "question": "What are the essential ingredients to prove murder under Section 302 IPC?",
        "reference": (
            "To prove murder under Section 302 IPC (read with Section 300 IPC), "
            "prosecution must establish: (1) death of a human being occurred; "
            "(2) death was caused by the accused's act; (3) the act was done with "
            "intention to cause death (mens rea); or intention to cause bodily injury "
            "likely to cause death; or knowledge that the act is imminently dangerous. "
            "Punishment: death or life imprisonment with fine."
        ),
        "key_acts": ["Section 302 IPC", "Section 300 IPC"],
    },

    # ── Verdict prediction (from facts) ───────────────────────
    {
        "id": "verd_01",
        "type": "verdict_prediction",
        "question": (
            "Based on the following facts, what would likely be the court's verdict?\n\n"
            "Facts: The accused issued a cheque of Rs. 5 lakhs to the complainant towards "
            "repayment of a loan. The cheque was dishonoured due to insufficient funds. "
            "The complainant sent a legal demand notice within 30 days. The accused did not "
            "repay within 15 days. The complaint was filed within 30 days of the notice period. "
            "Court: Chief Judicial Magistrate"
        ),
        "reference": (
            "The court would likely convict the accused under Section 138 of the Negotiable "
            "Instruments Act as all procedural requirements have been met: dishonour, "
            "timely notice, failure to repay, and timely complaint. "
            "Punishment may include imprisonment up to 2 years and/or fine up to Rs. 10 lakhs "
            "(twice the cheque amount). The complainant may also receive compensation."
        ),
        "key_acts": ["Section 138 Negotiable Instruments Act"],
    },
    {
        "id": "verd_02",
        "type": "verdict_prediction",
        "question": (
            "Based on the following facts, what would likely be the court's verdict?\n\n"
            "Facts: A tenant has been occupying a shop for 22 years without any written lease "
            "agreement. The landlord now wants to evict him citing personal need. "
            "The tenant claims adverse possession. The property is in Tamil Nadu. "
            "Court: District Civil Court"
        ),
        "reference": (
            "Adverse possession claim would likely fail as the tenant's possession was "
            "with the landlord's permission (permissive possession), which is a key "
            "disqualification. However, eviction would be governed by the Tamil Nadu "
            "Buildings (Lease and Rent Control) Act 1960 — landlord must prove genuine "
            "personal need. Courts in Tamil Nadu typically scrutinise personal need claims strictly."
        ),
        "key_acts": ["Tamil Nadu Buildings Lease and Rent Control Act 1960", "Limitation Act 1963"],
    },

    # ── Acts identification ────────────────────────────────────
    {
        "id": "acts_01",
        "type": "acts_cited",
        "question": "Which laws govern a dowry death case in India?",
        "reference": (
            "A dowry death case is governed by: "
            "Section 304B IPC (dowry death — 7 years to life imprisonment); "
            "Section 498A IPC (cruelty by husband/relatives); "
            "Section 113B of the Indian Evidence Act (presumption of dowry death); "
            "Dowry Prohibition Act 1961 (prohibits giving/taking dowry); "
            "Protection of Women from Domestic Violence Act 2005 (civil remedies)."
        ),
        "key_acts": ["Section 304B IPC", "Section 498A IPC", "Dowry Prohibition Act 1961"],
    },
    {
        "id": "acts_02",
        "type": "acts_cited",
        "question": "What acts and sections apply when a company fails to pay employee gratuity?",
        "reference": (
            "The Payment of Gratuity Act 1972 governs gratuity. "
            "Section 4 mandates gratuity after 5 years of continuous service at "
            "15 days wages per completed year. Section 7 governs the determination "
            "of gratuity. Non-payment is an offence under Section 9 (imprisonment 6 months "
            "to 2 years). Controlling Authority under Section 3 can be approached. "
            "Appeal lies to Appellate Authority under Section 7(7)."
        ),
        "key_acts": ["Payment of Gratuity Act 1972", "Section 4", "Section 9"],
    },
]

print(f'✅  Curated test suite: {len(CURATED_TEST)} questions')
type_dist = {}
for q in CURATED_TEST:
    type_dist[q["type"]] = type_dist.get(q["type"], 0) + 1
for t, n in type_dist.items():
    print(f'   {t:<25} {n} questions')

✅  Curated test suite: 19 questions
   legal_concept             15 questions
   verdict_prediction        2 questions
   acts_cited                2 questions


In [13]:
!pip install rouge-score bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=60ef07d10dcc94c7ca56faac8ce2c045530526d532ee21aa5801f623b1bccb6d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [14]:
# ── CELL 11 : Run evaluation ──────────────────────────────────
# Generates model answers for all test questions.
# Scores each with ROUGE-L, BERTScore, and Act Citation Accuracy.

from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
import pandas as pd
import re

FastLanguageModel.for_inference(model)

SYSTEM = (
    "You are LawGPT-IN, an expert AI assistant specialising in Indian law. "
    "You have deep knowledge of the Indian Penal Code, CrPC, Constitution of India, "
    "and landmark Supreme Court and High Court judgements. "
    "Always cite the relevant acts and sections in your answers."
)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)


def generate_answer(question, max_new_tokens=400):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            use_cache=True, temperature=0.1,  # low temp for eval consistency
            do_sample=False,                   # greedy for reproducibility
        )

    raw = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
    if "<|im_start|>assistant\n" in raw:
        answer = raw.split("<|im_start|>assistant\n")[-1]
        answer = answer.replace("<|im_end|>", "").strip()
    else:
        answer = raw.strip()
    return answer


def check_act_citations(prediction, key_acts):
    """
    Check what % of expected acts/sections appear in the model's answer.
    Partial string match — catches 'Section 438' even if CrPC suffix varies.
    """
    if not key_acts:
        return 1.0
    pred_lower = prediction.lower()
    hits = sum(
        1 for act in key_acts
        if any(part.lower() in pred_lower for part in act.split() if len(part) > 3)
    )
    return hits / len(key_acts)


# ── Generate all predictions ───────────────────────────────────
print(f'🤖  Running inference on {len(CURATED_TEST)} questions...\n')
results = []

for i, item in enumerate(CURATED_TEST):
    print(f'   [{i+1:02d}/{len(CURATED_TEST)}] {item["id"]}...', end=' ')
    prediction = generate_answer(item["question"])

    # ROUGE-L
    rouge_l = scorer.score(item["reference"], prediction)["rougeL"].fmeasure

    # Act citation accuracy
    act_acc = check_act_citations(prediction, item.get("key_acts", []))

    results.append({
        "id":         item["id"],
        "type":       item["type"],
        "question":   item["question"][:80] + "...",
        "reference":  item["reference"],
        "prediction": prediction,
        "rouge_l":    round(rouge_l, 4),
        "act_acc":    round(act_acc, 4),
    })
    print(f'ROUGE-L={rouge_l:.3f}  ActAcc={act_acc:.2f}')

# ── BERTScore (batch — faster) ─────────────────────────────────
print('\n⏳  Computing BERTScore (batch)...')
references  = [r["reference"]  for r in results]
predictions = [r["prediction"] for r in results]

_, _, bert_f1s = bert_score_fn(
    predictions, references,
    lang="en", verbose=False, device="cuda"
)
for i, f1 in enumerate(bert_f1s):
    results[i]["bert_f1"] = round(f1.item(), 4)

print('✅  BERTScore computed.')

🤖  Running inference on 19 questions...

   [01/19] bail_01... 

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:254: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ROUGE-L=0.211  ActAcc=1.00
   [02/19] bail_02... 

Both `max_new_tokens` (=400) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ROUGE-L=0.123  ActAcc=0.00
   [03/19] prop_01... 

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅  BERTScore computed.


In [15]:
# ── CELL 12 : Results summary ─────────────────────────────────
import pandas as pd

df = pd.DataFrame(results)

print('='*60)
print('  LawGPT-IN  ·  Evaluation Results')
print('='*60)

# Overall
print('\n📊  Overall Scores:')
print(f'   ROUGE-L (avg)       : {df["rouge_l"].mean():.4f}  (target > 0.30)')
print(f'   BERTScore F1 (avg)  : {df["bert_f1"].mean():.4f}  (target > 0.85)')
print(f'   Act Citation Acc    : {df["act_acc"].mean():.4f}  (target > 0.50)')

# Per type
print('\n📋  Scores by question type:')
print(df.groupby('type')[['rouge_l','bert_f1','act_acc']].mean().round(4).to_string())

# Best and worst
print('\n🏆  Top 3 (by BERTScore):')
top3 = df.nlargest(3, 'bert_f1')[['id','type','rouge_l','bert_f1','act_acc']]
print(top3.to_string(index=False))

print('\n⚠️   Bottom 3 (by BERTScore) — review these:')
bot3 = df.nsmallest(3, 'bert_f1')[['id','type','rouge_l','bert_f1','act_acc']]
print(bot3.to_string(index=False))

# Full table
print('\n📄  Full results table:')
print(df[['id','type','rouge_l','bert_f1','act_acc']].to_string(index=False))

  LawGPT-IN  ·  Evaluation Results

📊  Overall Scores:
   ROUGE-L (avg)       : 0.2544  (target > 0.30)
   BERTScore F1 (avg)  : 0.8622  (target > 0.85)
   Act Citation Acc    : 0.8421  (target > 0.50)

📋  Scores by question type:
                    rouge_l  bert_f1  act_acc
type                                         
acts_cited           0.2397   0.8590      1.0
legal_concept        0.2606   0.8641      0.8
verdict_prediction   0.2226   0.8516      1.0

🏆  Top 3 (by BERTScore):
      id          type  rouge_l  bert_f1  act_acc
 prop_02 legal_concept   0.3448   0.9001      1.0
const_01 legal_concept   0.4246   0.8988      1.0
 cons_01 legal_concept   0.4384   0.8972      1.0

⚠️   Bottom 3 (by BERTScore) — review these:
     id          type  rouge_l  bert_f1  act_acc
 lab_01 legal_concept   0.1415   0.8215      0.5
crim_02 legal_concept   0.2345   0.8309      1.0
prop_01 legal_concept   0.1577   0.8389      1.0

📄  Full results table:
      id               type  rouge_l  bert_f1  

In [16]:
# ── CELL 13 : Inspect individual answers ──────────────────────
# Change question_id below to read any result in full.

question_id = "bail_01"   # ← change to any id from the results table

row = next((r for r in results if r["id"] == question_id), None)
if not row:
    print(f'ID "{question_id}" not found. Available: {[r["id"] for r in results]}')
else:
    print(f'ID      : {row["id"]}')
    print(f'Type    : {row["type"]}')
    print(f'ROUGE-L : {row["rouge_l"]}')
    print(f'BERTScore F1 : {row["bert_f1"]}')
    print(f'Act Acc : {row["act_acc"]}')
    print('\n' + '─'*60)
    print('QUESTION:')
    print(CURATED_TEST[next(i for i,q in enumerate(CURATED_TEST) if q['id']==question_id)]['question'])
    print('\n' + '─'*60)
    print('REFERENCE ANSWER:')
    print(row['reference'])
    print('\n' + '─'*60)
    print('MODEL ANSWER:')
    print(row['prediction'])
    print('─'*60)

ID      : bail_01
Type    : legal_concept
ROUGE-L : 0.2105
BERTScore F1 : 0.8615
Act Acc : 1.0

────────────────────────────────────────────────────────────
QUESTION:
What is the difference between Anticipatory Bail and Regular Bail under CrPC?

────────────────────────────────────────────────────────────
REFERENCE ANSWER:
Anticipatory Bail is granted under Section 438 CrPC before arrest, when a person apprehends arrest for a non-bailable offence. Regular Bail is granted under Section 437 or 439 CrPC after arrest and during the pendency of trial. Key difference: anticipatory bail is pre-arrest protection; regular bail is post-arrest release.

────────────────────────────────────────────────────────────
MODEL ANSWER:
Anticipatory Bail and Regular Bail are two different types of bail provisions under the CrPC.

Regular Bail: It is granted after arrest. A person who has been arrested and is in custody can apply for regular bail. The court considers the merits of the case and grants bail i

In [17]:
# ── CELL 14 : Save results ─────────────────────────────────────

results_path = "lawgpt_eval_results.csv"
df.to_csv(results_path, index=False)
print(f'✅  Results saved to {results_path}')

# Also save as JSON for programmatic use
import json
with open("lawgpt_eval_results.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('✅  Results saved to lawgpt_eval_results.json')

# Download files to your local machine
from google.colab import files
files.download(results_path)
files.download("lawgpt_eval_results.json")

print('\n🎯  Next step: Stage 3 — Hybrid RAG + Web UI')

✅  Results saved to lawgpt_eval_results.csv
✅  Results saved to lawgpt_eval_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎯  Next step: Stage 3 — Hybrid RAG + Web UI
